In [6]:
from datasets import load_dataset, Dataset
import re
import pandas as pd
from typing import List, Dict
import json

In [35]:
# raw_data = "project-31-at-2026-01-04-19-00-b7bc78b3.json"

In [32]:
# def build_task_document_map(json_path):
#     with open(json_path, "r", encoding="utf-8") as f:
#         data = json.load(f)

#     task_doc_map = {}

#     for task in data:
#         task_id = task.get("source_task_id") or task.get("id")

#         # 这里确认是 data["text"]
#         document = task.get("data", {}).get("text")

#         if task_id is not None and document is not None:
#             task_doc_map[task_id] = document

#     return task_doc_map


In [33]:
# id2doc = build_task_document_map(raw_data)

In [34]:
# id2doc[2356]

In [3]:
data = load_dataset("TheFinAI/JF-TE", split="train",trust_remote_code=True)

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
data_df = data.to_pandas()

In [5]:
data_df

,id,text,entities,meta
0,0,(注) 1．当社は中間連結財務諸表を作成しているので、提出会社の主要な経営指標等の推移につい...,"[{'start': 9, 'end': 17, 'label': 'FinanceTerm...","{'source_task_id': 2356, 'note_header': '【Note..."
1,1,(注3) 主に通信販売している機能性表示食品「ごま豆乳仕立てのみんなのみかたDHA」、特定保...,"[{'start': 100, 'end': 109, 'label': 'FinanceT...","{'source_task_id': 2356, 'note_header': '【Note..."
2,2,(注）2024年4月17日付で公衆の縦覧に供されている変更報告書において、野村證券株式会社及...,"[{'start': 27, 'end': 32, 'label': 'FinanceTer...","{'source_task_id': 2356, 'note_header': '【Note..."
3,3,"(注) 1 「完全議決権株式（その他）」欄の普通株式には、証券保管振替機構名義の株式 5,0...","[{'start': 7, 'end': 19, 'label': 'FinanceTerm...","{'source_task_id': 2356, 'note_header': '【Note..."
4,4,"(注) 1 株主名簿上は、当社名義となっていますが、実質的に所有していない株式が1,000株...","[{'start': 72, 'end': 77, 'label': 'FinanceTer...","{'source_task_id': 2356, 'note_header': '【Note..."
...,...,...,...,...
197,197,（注） 短期借入金は、1年内返済予定の長期借入金を含めております。\n当連結会計年度 契約上...,"[{'start': 4, 'end': 9, 'label': 'FinanceTerm'...","{'source_task_id': 2377, 'note_header': '【Note..."
198,198,(注）1．当該役員が一身上の都合により任期途中で退任(2023年12月8日)したことから、そ...,"[{'start': 1291, 'end': 1299, 'label': 'Financ...","{'source_task_id': 2377, 'note_header': '【Note..."
199,199,（注）1．非支配持分は、取得日における被取得企業の識別可能な純資産の公正価値に対\nする非支...,"[{'start': 63, 'end': 66, 'label': 'FinanceTer...","{'source_task_id': 2377, 'note_header': '【Note..."
200,200,（注）上表に含まれない市場価格のない株式等の貸借対照表計上額\n貸借対照表計上額\n関連会社...,"[{'start': 40, 'end': 47, 'label': 'FinanceTer...","{'source_task_id': 2377, 'note_header': '【Note..."


In [7]:
def build_financial_term_groups(text: str, spans: List[Dict]) -> List[List[str]]:
    """
    给定原始文本和 span 列表，构造 gold JSON list-of-lists。
    
    spans: 形如 [{"start": int, "end": int, ...}, ...]
           label 字段会被忽略，只用 start/end.
           
    返回值: List[List[str]]
        [
          [max_term_1, nested_1_1, nested_1_2, ...],
          [max_term_2],
          ...
        ]
    """
    # 只保留 start/end，并做一下排序（先按 start，后按 -end 保证长的在前）
    cleaned_spans = [
        {"start": s["start"], "end": s["end"]}
        for s in spans
    ]
    cleaned_spans.sort(key=lambda x: (x["start"], -x["end"]))

    # 1. 找出 maximal spans（不被任何其它 span 严格包含的 span）
    maximal_spans = []
    for i, s in enumerate(cleaned_spans):
        s_start, s_end = s["start"], s["end"]
        contained = False
        for j, t in enumerate(cleaned_spans):
            if i == j:
                continue
            t_start, t_end = t["start"], t["end"]
            # t 严格包含 s：左边不大于，右边不小于，且至少一端严格
            if t_start <= s_start and t_end >= s_end and (t_start < s_start or t_end > s_end):
                contained = True
                break
        if not contained:
            maximal_spans.append(s)

    # 2. 对每个 maximal span，收集它的所有 nested spans（包括自身）
    groups = []
    for max_span in maximal_spans:
        m_start, m_end = max_span["start"], max_span["end"]
        nested = []
        for s in cleaned_spans:
            s_start, s_end = s["start"], s["end"]
            if m_start <= s_start and s_end <= m_end:
                nested.append(s)

        # 去重 + 按长度降序排序（长的更像“专用术语”）
        # 也可以改成按 start 排序：sorted(nested, key=lambda x: (x["start"], -x["end"]))
        nested_unique = []
        seen = set()
        for s in nested:
            key = (s["start"], s["end"])
            if key not in seen:
                seen.add(key)
                nested_unique.append(s)

        nested_unique.sort(key=lambda x: (-(x["end"] - x["start"]), x["start"]))

        # 3. 把 span 映射回文本片段
        terms = [text[s["start"]:s["end"]] for s in nested_unique]
        groups.append(terms)

    return groups

In [62]:
querys = []
answers = []
entity_len = []
for i in range(len(data_df)):
    text = data_df.at[i,"text"]
    entities = data_df.at[i, "entities"]
    meta = data_df.at[i, "meta"]

    # doc_id = meta.get("source_task_id")
    # doc_text = id2doc[doc_id]
    
    answer = build_financial_term_groups(text,entities)
    
    query = get_query(text)

    querys.append(query)
    answers.append(answer)
    entity_len.append(len(entities))

In [63]:
max(entity_len)

78

In [64]:
min(entity_len)

1

In [66]:
sum(entity_len)/len(entity_len)

11.94059405940594

In [52]:
new_df = pd.DataFrame({"query":querys, "answer":answers})

In [53]:
new_df.at[1,"answer"]

[['キャッシュ・フロー'],
 ['財務活動によるキャッシュ・フロー'],
 ['税金等調整前中間純利益', '中間純利益', '純利益'],
 ['減価償却費'],
 ['未払費用'],
 ['有形固定資産'],
 ['中間連結会計期間', '連結会計期間', '会計期間'],
 ['研究開発']]

In [54]:
new_df.at[0,"query"]

'You are a Japanese financial expert specializing in financial disclosure analysis.\nRead the following Japanese financial note carefully.\n\nYour task is to extract financial terms and organize them by structure.\nDefinitions:\n    * Maximal financial term\n        - A financial term that is not fully contained inside any longer financial term.\n    * Nested financial term\n        - A financial term that appears inside a maximal financial term.\n\nWhat to extract:\n    * Identify all maximal financial terms in the text.\n    * For each maximal term, identify all nested financial terms it contains.\n    * Include only domain-specific financial terminology. Do NOT include generic expressions or normal language.\n    \nOutput format (STRICT):\n    * Return a JSON list of lists, each inner list corresponds to one maximal financial term:\n        - The first element is the maximal term.\n        - The remaining elements (if any) are its nested financial terms.\n        - If a maximal term

In [55]:
from huggingface_hub import HfApi

In [56]:
hf_dataset = Dataset.from_pandas(new_df, preserve_index=True)
hf_dataset = hf_dataset.rename_column("__index_level_0__", "id")

In [59]:
dataset_repo = "TheFinAI/JFTE"

In [60]:
hf_dataset.push_to_hub(dataset_repo,split="test",private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/TheFinAI/JFTE/commit/177a21190d68d9539004f25ac10580a7051f9d09', commit_message='Upload dataset', commit_description='', oid='177a21190d68d9539004f25ac10580a7051f9d09', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/TheFinAI/JFTE', endpoint='https://huggingface.co', repo_type='dataset', repo_id='TheFinAI/JFTE'), pr_revision=None, pr_num=None)